# Probability Distributions & Sampling

## Definition
Probability distributions describe how values are spread. NumPy's `random.Generator`
provides functions to sample from many standard distributions.

### Available Distributions
| Distribution | Function | Parameters | AI/ML Use |
|-------------|----------|------------|----------|
| Uniform | `rng.uniform(low, high)` | — | Random init, data augment |
| Normal | `rng.normal(loc, scale)` | μ, σ | Xavier init, noise |
| Standard Normal | `rng.standard_normal()` | — | He init |
| Exponential | `rng.exponential(scale)` | λ | Inter-arrival times |
| Binomial | `rng.binomial(n, p)` | n, p | Bernoulli trials |
| Bernoulli | `rng.binomial(1, p)` | p | Dropout mask |
| Poisson | `rng.poisson(lam)` | λ | Count data |
| Beta | `rng.beta(a, b)` | α, β | Mixup augmentation |
| Gamma | `rng.gamma(shape, scale)` | — | Bayesian priors |
| Dirichlet | `rng.dirichlet(alpha)` | α | Topic modeling |
| Multinomial | `rng.multinomial(n, pvals)` | — | Language sampling |
| Laplace | `rng.laplace(loc, scale)` | μ, b | Sparse priors |
| Cauchy | `rng.standard_cauchy()` | — | Heavy tails |

In [ ]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

rng = np.random.default_rng(42)

# Sample from several distributions
normal    = rng.normal(loc=0, scale=1, size=10000)
uniform   = rng.uniform(low=-3, high=3, size=10000)
exponential = rng.exponential(scale=1.0, size=10000)

print('Normal:      mean=', round(normal.mean(), 3), ' std=', round(normal.std(), 3))
print('Uniform:     mean=', round(uniform.mean(), 3), ' std=', round(uniform.std(), 3))
print('Exponential: mean=', round(exponential.mean(), 3))

In [ ]:
# Normal distribution in depth
rng = np.random.default_rng(42)
samples = rng.normal(loc=0, scale=1, size=100000)

print('68% within 1 std:', np.mean(np.abs(samples) <= 1))
print('95% within 2 std:', np.mean(np.abs(samples) <= 2))
print('99% within 3 std:', np.mean(np.abs(samples) <= 3))

In [ ]:
# He initialization (for ReLU networks)
# He normal: std = sqrt(2 / fan_in)
def he_init(fan_in, fan_out, rng):
    std = np.sqrt(2.0 / fan_in)
    return rng.normal(0, std, size=(fan_in, fan_out))

rng = np.random.default_rng(42)
W = he_init(512, 256, rng)
print('He init shape:', W.shape)
print('Std:', np.round(np.std(W), 4), '  Expected:', np.round(np.sqrt(2.0/512), 4))

In [ ]:
# Bernoulli / Binomial — Dropout mask
rng = np.random.default_rng(42)
dropout_rate = 0.2  # drop 20% of neurons
batch_size, features = 4, 8

# Bernoulli mask: keep with prob (1 - dropout_rate)
mask = rng.binomial(1, 1 - dropout_rate, size=(batch_size, features))
activations = rng.standard_normal((batch_size, features))

# Apply dropout
out = activations * mask / (1 - dropout_rate)  # inverted dropout
print('Dropout mask:\n', mask)
print('Zeroed out fraction:', 1 - mask.mean())

In [ ]:
# Beta distribution — Mixup augmentation
rng = np.random.default_rng(42)
alpha = 0.4  # Mixup hyperparameter

# Sample mixing coefficients
lambdas = rng.beta(alpha, alpha, size=5)
print('Mixup lambda samples:', np.round(lambdas, 3))
# lambda close to 0 or 1 means one image dominates (with alpha < 1)

In [ ]:
# Dirichlet distribution — probability vectors
rng = np.random.default_rng(42)

# Dirichlet gives a valid probability vector (sums to 1)
# Used in topic models like LDA
alpha = np.array([1.0, 2.0, 3.0])
samples = rng.dirichlet(alpha, size=5)
print('Dirichlet samples (each row sums to 1):')
print(np.round(samples, 3))
print('Row sums:', samples.sum(axis=1))

In [ ]:
# Multinomial — simulating categorical sampling
rng = np.random.default_rng(42)

# Probabilities of 5 words in vocab
probs = np.array([0.1, 0.3, 0.2, 0.25, 0.15])
# Simulate picking 1 token 10 times
counts = rng.multinomial(n=10, pvals=probs)
print('Word counts in 10 draws:', counts)
print('Sum:', counts.sum())

In [ ]:
# Poisson — modeling arrival rates
rng = np.random.default_rng(42)
lam = 3.5  # average events per unit time
arrivals = rng.poisson(lam=lam, size=1000)
print(f'Poisson(lambda={lam}):')
print(f'  Theoretical mean: {lam}')
print(f'  Sample mean:      {arrivals.mean():.3f}')
print(f'  Sample var:       {arrivals.var():.3f}')  # Poisson: mean == var

In [ ]:
# Laplace distribution — sparse priors (L1 regularization)
rng = np.random.default_rng(42)
laplace = rng.laplace(loc=0, scale=1, size=1000)
print('Laplace: heavier tails than Normal')
print('  Kurt:', round(float(((laplace - laplace.mean())**4).mean() / laplace.std()**4), 2), '(Normal kurtosis = 3)')